# Cloud-Based Image Processing - Exercises

_Mikołaj Leszczuk, Szymon Turek, Jakub Nawała_

![](https://i.creativecommons.org/l/by/4.0/88x31.png)

## Setup Tasks

Below are instructions for setting up an environment for computer vision applications (Python + image processing libraries).

**Python** is a general-purpose high-level programming language with an extensive standard library package, whose guiding principle is the readability and clarity of the source code. Its syntax is characterized by transparency and brevity. 

**Clarifai** is an Artificial Intelligence (AI) cloud service that specializes in computer vision and uses machine learning and deep neural networks to identify and analyze images and videos. The cloud service offers its solutions via API, mobile SDK, and on-premises solutions.

### App-Specific API Keys

App-specific API Keys are used to authorize your Clarifai applications. A key is automatically generated when you create a new application. You can also go to the application's list, select an app of your choice and create a new key in the app details page. _Each API key is tied to a specific user and a specific app._

After creating a free account, the next steps are: (i) create a new application and (ii) generate the application key (the key should be put in the script).

### Create an Application and Generate the Application Key

To create an application, head on over to the applications page and press the 'Create Application' button.

![](https://gblobscdn.gitbook.com/assets%2F-LisemUaXxC3S7Kk9xFz%2Fsync%2F069fd59ae3fb51f51b3acbed9dd72e997287a0f2.png?alt=media)

Start by creating a free Clarifai account ([https://portal.clarifai.com/signup](https://portal.clarifai.com/signup)).

[`key.txt`](key.txt)

### API Client Installation Instructions

Using client library to access API.

**Tip 1**: Please note that if you use Python, there might be two independently configured versions installed on the machine: Python 2 (`python`) and Python 3 (`python3`). The two versions are not always compatible. You are advised to use Python 3 as Python 2 support officially ended in January 2020.

**Tip 2**: To install Clarifai for Python, you must execute the following _pip_ command (usually you do not need administrative privileges to run this command, please keep in mind that the _pip_ command installs packages only for Python 3):

In [ ]:
pip install clarifai-grpc

**Tip 3**: In case of error messages saying that the pip version is too old, you must upgrade it by executing the following command (it can be noted that this should not be necessary and may break your machine configuration):

In [ ]:
pip install --upgrade pip

**Tip 4**: In case of error messages saying that you do not have sufficient permissions, you can use the root (administrator) account. **Before you do so, please first consult with the tutor.**

**Tip 5**: **If you get the following error when installing the library: ``Failed building wheel for grpcio``**

Try upgrading **setuptools** to a version **40.7.1** or higher.

In [ ]:
pip install --upgrade setuptools

## Example - Tags Extraction on Images via URL (Prediction of Concepts in an Image)

Construct the `V2Stub` object using which you'll access all the Clarifai API functionality:

In [ ]:
from clarifai_grpc.channel.clarifai_channel import ClarifaiChannel
from clarifai_grpc.grpc.api import service_pb2_grpc

stub = service_pb2_grpc.V2Stub(ClarifaiChannel.get_grpc_channel())

Predict concepts in an image:

![](https://samples.clarifai.com/metro-north.jpg)

In [ ]:
from clarifai_grpc.grpc.api import service_pb2, resources_pb2
from clarifai_grpc.grpc.api.status import status_code_pb2

This is how you authenticate. This will be used by every Clarifai endpoint call.

In [ ]:
with open('key.txt') as f:
    key = f.read()

metadata = (('authorization', 'Key '+key),)

We specify that the model belongs to clarifai/main.

In [ ]:
USER_ID = "clarifai"
APP_ID  = "main"
userDataObject = resources_pb2.UserAppIDSet(user_id=USER_ID, app_id=APP_ID)

This is the model ID of a publicly available General model. You may use any other public or custom model ID. We use the public model "general-image-recognition".

In [ ]:
MODEL_ID = "general-image-recognition"

The script should point the Clarifai API to a URL path to an image:

In [ ]:
request = service_pb2.PostModelOutputsRequest(
    user_app_id=userDataObject,
    model_id=MODEL_ID,
    inputs=[
        resources_pb2.Input(
            data=resources_pb2.Data(
                image=resources_pb2.Image(
                    url="https://samples.clarifai.com/metro-north.jpg"
                )
            )
        )
    ]
)

response = stub.PostModelOutputs(request, metadata=metadata)

Finally, please print the output data:

In [ ]:
print(response)

As the response is formatted using JSON (JavaScript Object Notation), instead of the regular **`print`**, it is better to use:

In [ ]:
for concept in response.outputs[0].data.concepts:
    print('%21s: %.2f' % (concept.name, concept.value))

Let's compare with the image...

In [ ]:
[print(concept.name, end=' ') for concept in response.outputs[0].data.concepts];

![](https://samples.clarifai.com/metro-north.jpg)

Let's try with "Lenna"...
![](https://upload.wikimedia.org/wikipedia/en/7/7d/Lenna_%28test_image%29.png)

In [ ]:
request = service_pb2.PostModelOutputsRequest(
    user_app_id=userDataObject,
    model_id=MODEL_ID,
    inputs=[
        resources_pb2.Input(
            data=resources_pb2.Data(
                image=resources_pb2.Image(
                    url="https://upload.wikimedia.org/wikipedia/en/7/7d/Lenna_%28test_image%29.png"
                )
            )
        )
    ]
)

response = stub.PostModelOutputs(request, metadata=metadata)

In [ ]:
[print(concept.name, end=' ') for concept in response.outputs[0].data.concepts];

![](https://upload.wikimedia.org/wikipedia/en/7/7d/Lenna_%28test_image%29.png)

Error handling...

In [ ]:
request = service_pb2.PostModelOutputsRequest(
    user_app_id=userDataObject,
    model_id=MODEL_ID,
    inputs=[
        resources_pb2.Input(
            data=resources_pb2.Data(
                image=resources_pb2.Image(
                    url="http://foo.bar"
                )
            )
        )
    ]
)

response = stub.PostModelOutputs(request, metadata=metadata)

In [ ]:
print(response)

In [ ]:
if response.status.code != status_code_pb2.SUCCESS:
    print("There was an error with your request!")
    print("\tCode: {}".format(response.outputs[0].status.code))
    print("\tDescription: {}".format(response.outputs[0].status.description))
    print("\tDetails: {}".format(response.outputs[0].status.details))
    raise Exception("Request failed, status code: " + str(response.status.code))